# درس ۰۵ - آر‌ای‌جی عامل محور


## راه‌اندازی

این دفترچه الگو الگوی Agentic RAG (تولید افزوده شده با بازیابی) را با استفاده از Microsoft Agent Framework نشان می‌دهد.

**پیش‌نیازها:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — نقطه پایانی سرویس جستجوی Azure AI شما
- `AZURE_SEARCH_API_KEY` — کلید API سرویس جستجوی Azure AI شما
- استقرار Azure OpenAI پیکربندی شده از طریق متغیرهای محیطی
- احراز هویت Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## RAG عاملی چیست؟

RAG سنتی از یک فرآیند ثابت پیروی می‌کند: ابتدا اسناد را بازیابی کرده، سپس پاسخ تولید می‌کند. **RAG عاملی** پا را فراتر گذاشته و به عامل استقلال می‌دهد تا تصمیم بگیرد **چه زمانی** و **چگونه** اطلاعات را بازیابی کند.

با RAG عاملی، عامل می‌تواند:
- **تصمیم بگیرد** که آیا قبل از پاسخ به سوال نیاز به بازیابی وجود دارد یا نه
- **انتخاب کند** که کدام منبع داده یا ابزار را برای پرس و جو استفاده کند
- **ارزیابی کند** نتایج بازیابی‌شده را و در صورت ناکافی بودن تلاش اول، بازیابی‌های بعدی انجام دهد
- **ترکیب کند** اطلاعات از چند مرحله بازیابی برای ارائه پاسخ یکپارچه

این باعث می‌شود عامل نسبت به فرآیند ثابت بازیابی-سپس-تولید، انعطاف‌پذیرتر و دقیق‌تر باشد.


## ایجاد یک ابزار جستجو

در Agentic RAG، منابع داده خارجی به عنوان **ابزارها** بسته‌بندی می‌شوند که عامل می‌تواند به صورت درخواستی فراخوانی کند. این امکان را به عامل می‌دهد که واکشی را فقط به عنوان یک اقدام دیگر که می‌تواند انجام دهد در نظر بگیرد، نه یک مرحله اجباری.

در ادامه، یک پایگاه دانش سفر تعریف می‌کنیم و آن را به عنوان ابزاری در دسترس عامل قرار می‌دهیم تا اطلاعات مقصد را جستجو کند.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## ساخت عامل RAG

اکنون یک عامل ایجاد می‌کنیم که به آن دستور داده شده است **همیشه قبل از پاسخ دادن اطلاعات را دریافت کند**. این عامل از ابزار `search_travel_knowledge` استفاده می‌کند تا پاسخ‌های خود را بر پایه پایگاه دانش قرار دهد به جای تکیه بر داده‌های آموزشی خود.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## بازیابی تکراری — الگوی سازنده-بازرس

یکی از مزایای کلیدی Agentic RAG، **بازیابی تکراری** است. عامل می‌تواند چندین دور جستجو انجام دهد تا یافته‌های اولیه خود را تأیید، اصلاح یا گسترش دهد — مشابه یک فرایند کاری «سازنده-بازرس»:

1. **مرحله سازنده**: عامل اطلاعات اولیه را بازیابی کرده و پیش‌نویس پاسخ را تهیه می‌کند.
2. **مرحله بازرس**: عامل بازیابی‌های اضافی را برای تأیید جزئیات یا پر کردن خلأها انجام می‌دهد.

در ادامه، از عامل سوالی پرسیده شده که نیاز به مقایسه چند مقصد دارد و باعث می‌شود چندین بار جستجو کند.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصه

در این درس یاد گرفتید چگونه یک سیستم **Agentic RAG** با استفاده از چارچوب Microsoft Agent بسازید:

- **Agentic RAG** به عوامل اجازه می‌دهد به صورت خودمختار تصمیم بگیرند چه زمانی اطلاعات را بازیابی کنند، و این باعث می‌شود بازیابی پویا باشد نه ثابت.
- **ابزارها به عنوان منابع داده**: پایگاه‌های دانش خارجی (مثل Azure AI Search) به عنوان ابزارهایی که عامل می‌تواند فراخوانی کند بسته‌بندی می‌شوند.
- **بازیابی تکراری**: الگوی سازنده-بازرس به عامل امکان می‌دهد چندین دور بازیابی انجام دهد — جستجو، تایید، و بازبینی — قبل از ارائه پاسخ نهایی.

در تولید، شما `TRAVEL_KNOWLEDGE_BASE` حافظه‌ای را با یک شاخص واقعی Azure AI Search جایگزین می‌کنید تا بازیابی اسناد سفر در مقیاس بزرگ را مدیریت کند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح ضروری**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است دارای اشکال یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود منبع معتبر محسوب می‌شود. برای اطلاعات حیاتی، توصیه می‌شود از ترجمه حرفه‌ای انسانی استفاده کنید. ما مسئول هیچ گونه سوءتفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
